# POLITE — device palette

Independent cells for connecting, inspecting, and deliberately moving each component. Run the shared preamble, then pull only the cells needed tonight.


## Preamble


In [ ]:
import os, sys
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / 'pyproject.toml').exists() and _root.parent != _root: _root = _root.parent
if not (_root / 'pyproject.toml').exists(): raise RuntimeError('Could not locate the POLITE repository root')
os.chdir(_root)
if str(_root) not in sys.path: sys.path.insert(0, str(_root))
print('POLITE root:', _root)


In [ ]:
from pathlib import Path
from obs_utils import live
from obs_utils import interactive as obs
from obs_utils import user_config as uc
SESSION_DIR = None  # set in Tonight's card


## Assembly status


In [ ]:
s = obs.connect_all()
s.status()


In [ ]:
from obs_utils.timing import acquire_session_timing_snapshot
timing = acquire_session_timing_snapshot()
print(timing)


## Camera


In [ ]:
s = obs.connect_camera()
cam = s.camera
print('sensor:', cam.CameraXSize, 'x', cam.CameraYSize, '| ROI:', cam.StartX, cam.StartY, cam.NumX, cam.NumY)


In [ ]:
cam.BinX = cam.BinY = 1
cam.ReadoutMode, cam.Gain, cam.Offset = 5, 56, 20
assert (cam.BinX, cam.BinY, cam.ReadoutMode, cam.Gain, cam.Offset) == (1, 1, 5, 56, 20)


In [ ]:
from obs_utils.night_safety import cooler_gate
cam.SetCCDTemperature = -15.0
cooler_gate(cam, -15.0, tol_c=0.5, stable_s=30.0, timeout_s=900.0)


In [ ]:
# MOTION — thermal ramp; only after all data are safe.
s.warm_up(rate_c_per_min=2.0)


In [ ]:
from obs_utils.roi import CameraROI, apply_roi
roi = CameraROI(startx=500, starty=0, numx=5280, numy=4210)
apply_roi(cam, roi)


In [ ]:
from obs_utils.roi import CameraROI, apply_roi
apply_roi(cam, CameraROI(startx=0, starty=0, numx=cam.CameraXSize, numy=cam.CameraYSize))


## Filter wheel


In [ ]:
from obs_utils.night_safety import INSTALLED_EFW_NAMES, verify_filter_wheel
s = obs.connect_filter_wheel()
verify_filter_wheel(s.imaging)
print(INSTALLED_EFW_NAMES)


In [ ]:
# MOTION
s.filter('Photometric V')
print(s.current_filter())


In [ ]:
# MOTION
s.filter(int(input('EFW slot (0-based): ')))
print(s.current_filter())


## HWP — Pyxis polarimetric rotator


In [ ]:
s = obs.connect_hwp()  # observatory Alpaca path
print('HWP:', s.hwp_rotator)


In [ ]:
# MOTION — serial path only; home once per power cycle.
s = obs.connect_hwp_serial()
s.home_hwp()


In [ ]:
# MOTION
from obs_utils.night_safety import HWP_DEFAULT_TOL_DEG
achieved = s.hwp(22.5)
assert abs(achieved - 22.5) <= HWP_DEFAULT_TOL_DEG


In [ ]:
# MOTION — one contiguous standard-angle exercise.
angles = (0.0, 22.5, 45.0, 67.5)
[(angle, s.hwp(angle)) for angle in angles]


## Field rotator — PWI4 instrument de-rotator, not the HWP


In [ ]:
s = obs.connect_field_rotator()
print(s.field_rotator_status())


In [ ]:
# MOTION
s.field_rotator_goto_field(45.0)
print(s.field_rotator_status())


In [ ]:
# MOTION
s.field_rotator_goto_mech(float(input('Mechanical angle (deg): ')))


In [ ]:
# MOTION
s.field_rotator_offset(float(input('Mechanical jog (deg): ')))


In [ ]:
# MOTION — emergency stop for the field rotator.
s.field_rotator_stop()


## Focuser — PWI4


In [ ]:
s = obs.connect_focuser()
print(s.focuser_status())


In [ ]:
FOCUSER_POSITION = float(input('Measured safe focuser position: '))


In [ ]:
# MOTION
s.focus(FOCUSER_POSITION)


In [ ]:
# MOTION
s.focus_relative(float(input('Focuser jog (steps): ')))


In [ ]:
# MOTION — emergency stop for the focuser.
s.focus_stop()


## Mount — PWI4; untested against a working DEC axis


In [ ]:
s = obs.connect_mount()
print(s.status(pretty=False)['mount'])


In [ ]:
# MOTION
from obs_utils.mount import enable_motors
enable_motors(s.pwi4)


In [ ]:
# MOTION — only after axis-enable verification.
from obs_utils.mount import home_mount
home_mount(s.pwi4)


In [ ]:
# MOTION
s.pwi4.mount_tracking_on()


In [ ]:
# MOTION
s.pwi4.mount_tracking_off()


In [ ]:
# MOTION
from obs_utils.mount import slew_radec_j2000
slew_radec_j2000(s.pwi4, float(input('RA (hours): ')), float(input('Dec (deg): ')))


In [ ]:
# MOTION
from obs_utils.mount import slew_altaz
slew_altaz(s.pwi4, float(input('Altitude (deg): ')), float(input('Azimuth (deg): ')))


In [ ]:
# MOTION
s.pwi4.mount_offset(ra_add_arcsec=float(input('RA offset (arcsec): ')), dec_add_arcsec=float(input('Dec offset (arcsec): ')))


In [ ]:
# MOTION
s.pwi4.mount_spiral_offset_new(float(input('Spiral X step (arcsec): ')), float(input('Spiral Y step (arcsec): ')))


In [ ]:
# MOTION — emergency stop.
s.pwi4.mount_stop()


In [ ]:
# MOTION — park only after the data and observatory are safe.
s.pwi4.mount_park()


In [ ]:
from obs_utils.obs_math import airmass_kasten_young
print(airmass_kasten_young(s.pwi4.status().mount.altitude_degs))


In [ ]:
from obs_utils.block_log import moon_separation_deg
print(moon_separation_deg(float(input('RA (hours): ')), float(input('Dec (deg): '))))


## Disconnect


In [ ]:
obs.shutdown()
